# ArtDiffuse-RL Training

Train a small diffusion model on Midjourney text-to-image data.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/input/artdiffuse-rl/src')

import torch
from torch.utils.data import DataLoader
from dataset import MidjourneyDataset
from train import build_model, train, load_config

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')

In [ ]:
# Load config
config = load_config('/kaggle/input/artdiffuse-rl/configs/base.yaml')
print(config)

In [ ]:
# Create dataset
dataset = MidjourneyDataset(
    data_dir='/kaggle/input/midjourney-texttoimage',
    image_size=64,
    max_samples=50000
)

dataloader = DataLoader(
    dataset,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    num_workers=config['data']['num_workers'],
    pin_memory=True
)

print(f'Dataset size: {len(dataset)}')
print(f'Batches per epoch: {len(dataloader)}')

In [ ]:
# Build model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(config).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')
print(f'Model size: {total_params * 4 / 1024 / 1024:.1f} MB')

In [ ]:
# Train
model = train(
    model=model,
    dataloader=dataloader,
    config=config['training'],
    device=device,
    output_dir='/kaggle/working/models'
)